In [1]:
import os
import numpy as np
import json
import pickle as pkl
import matplotlib.pyplot as plt
import seaborn as sns
import skvideo.io
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle
import matplotlib.font_manager
import matplotlib as mpl
mpl.use('Agg')
params = {'font.family': 'serif','font.serif': 'Times', 'text.usetex': True,'mathtext.fontset': 'custom'}
mpl.rcParams.update(params)

In [2]:
def read_anno(anno_path):
    with open(anno_path, 'r') as f:
        anno = json.load(f)
    if 'database' in anno:
        anno = anno['database']
    if "results" in anno:
        anno = anno['results']
    return anno

In [3]:
video_root = '/mnt/cephfs/ec/home/chenzhuokun/git/swallowProject/result/datas'

In [4]:
result_path = '../tmp/best_result.json' # see tools/result_parser.ipynb to generate this file from raw pkl file
ground_truth_path = '../data/swallow/anno/swallow_singlestage_without_all_ana.json'

In [5]:
ruan_result_path = '../outputs/ckpt_swallow_stage2_lgte/final_result.json'
hyder_result_path = '../outputs/ckpt_swallow_2tower_10ep/final_result.json'

In [6]:
ruan_a2net_result_path = '/mnt/cephfs/home/liyirui/project/swallow_a2net_vswg/output/test_exp_zk2stage_20241127162039.txt'

In [7]:
def get_label_dict_from_gt(gt):
    '''
    Extract a mapping of action names (labels) to their corresponding IDs from the ground truth.
    
    Parameters:
    - gt: dict - Ground truth annotations.
    
    Returns:
    - label_dict: dict - A dictionary mapping action names (str) to their IDs (int).
    '''
    label_dict = {}
    
    # Iterate through the ground truth to extract unique labels and their IDs
    for video_name, data in gt.items():
        for annotation in data['annotations']:
            label = annotation['label']
            label_id = annotation['label_id']
            
            # Add the label and ID to the dictionary if not already present
            if label not in label_dict:
                label_dict[label] = label_id
    
    return label_dict


In [8]:
result = read_anno(result_path)
ground_truth = read_anno(ground_truth_path)

In [9]:
ruan_result = read_anno(ruan_result_path)
# hyder_result = read_anno(hyder_result_path)

In [10]:
label_dict = get_label_dict_from_gt(ground_truth)
reversed_label_dict = {v:k for k,v in label_dict.items()}
ruan_a2net_result = {}
with open(ruan_a2net_result_path,'r')as f:
    data = f.readlines()
for i in data:
    video_name, t_start, t_end, label_id, score = i.strip().split(" ")
    t_start, t_end, score = float(t_start), float(t_end), float(score)
    if video_name not in ruan_a2net_result:
        ruan_a2net_result[video_name] = [
            {
                'segment': [t_start, t_end],
                'score': score,
                'label': reversed_label_dict[int(label_id)%7]
            }
        ]
    else:
        ruan_a2net_result[video_name].append({
                'segment': [t_start, t_end],
                'score': score,
                'label': reversed_label_dict[int(label_id)%7]
            })

In [11]:
len(ruan_a2net_result)

137

In [12]:
results_dict = {
    'Ruan et al. (A2Net)': ruan_a2net_result,
    'Ruan et al. (ActionMamba)': ruan_result,
    'Ours (ActionMamba)': result,
}

In [13]:
vidoe_names = list(result.keys())

In [14]:
index = 33

In [15]:
video_name = vidoe_names[index]

In [16]:
video_name

'10_144.0_2021082501_liu2meng2_jian4kang1cha2ti3_2021_08_25_152802_64'

In [17]:
video_data = skvideo.io.vread(os.path.join(video_root, video_name+'.avi'))

In [18]:
dur = ground_truth[video_name]['duration']
dur

63.930533

In [19]:
def sample_videos(video_data, sample_index):
    return [video_data[i] for i in sample_index]

In [20]:
def plot_gt_n_result(video_name, ground_truth, result, dur, start_time, end_time, conf=0.5):
    fig, ax = plt.subplots(1, 1, figsize=(20, 5))
    ax.set_xlim([0, dur])
    ax.set_ylim([0, 1])
    ax.set_title(video_name)
    ax.set_xlabel('time (s)')
    ax.set_ylabel('score')
    ax.plot([0, dur], [0.5, 0.5], 'r--')
    if video_name in ground_truth:
        gt = ground_truth[video_name]
        for i in range(len(gt['annotations'])):
            start = gt['annotations'][i]['segment'][0]
            end = gt['annotations'][i]['segment'][1]
            ax.plot([start, end], [0.9, 0.9], 'g', label='ground truth')
    else:
        print('no ground truth')
    if video_name in result:
        res = result[video_name]
        for i in range(len(res)):
            start = res[i]['segment'][0]
            end = res[i]['segment'][1]
            score = res[i]['score']
            if score > conf:
                ax.plot([start, end], [score, score], 'b', label='result')
            else:
                continue
    plt.show()
    plt.close()

In [21]:
def gt_plot(video_data, ground_truth, action_num=7, video_name=None, start=0, end=-1):
    '''
    plot  C*T (action category * Time) matrix with ground truth
    video_data: np.array (T,H,W,3)
    ground_truth: 
        {
                $video_name:{
                    annotations:
                    [
                        {
                            segment: [start_time, end_time],
                            label: action_name,
                            label_id: id
                        },...
                    ]
                }
        }
    '''
    T = video_data.shape[0]  # Number of frames
    C = action_num  # Number of action categories (assuming action_name is the highest category id)
    
    # Initialize the C*T matrix with zeros
    gt_matrix = np.zeros((C, T))
    fps = 29.97002997002997
    if video_name is None:
        # Extract the video name (assuming the video name is the key in the ground truth dictionary)
        video_name = list(ground_truth.keys())[0]
        
    # Fill the matrix with 1s where the action is occurring
    for annotation in ground_truth[video_name]['annotations']:
        start_time = annotation['segment'][0]
        end_time = annotation['segment'][1]
        label_id = annotation['label_id']
        
        # Ensure the times are within the video length
        start_time = max(0, min(start_time, T-1))
        end_time = max(0, min(end_time, T-1))
        
        start_frame = round(fps * start_time)
        end_frame = round(fps * end_time)
        gt_matrix[label_id, start_frame:end_frame+1] = 1
    
    # Plot the matrix
    plt.figure(figsize=(10, 6))
    sns.heatmap(gt_matrix[:,start:end])
    plt.xlabel('Time (frames)')
    plt.ylabel('Action Category')
    plt.yticks([])
    plt.title(f'Ground Truth Action Matrix for {video_name}')
    plt.show()
    plt.close()


In [22]:
action_names = ['LaryngealVestibuleClosure', 'UESOpen', 'OralDelivery', 'ThroatTransport', 'HyoidExercise', 'ThroatSwallow', 'SoftPalateLift']
paper_action_names = [
    'Laryngeal\nVestibule\nClosure',
    'UES\nOpening',
    'Oral\nTransit',
    'Pharyngeal\nTransit',
    'Hyoid\nMotion',
    'Swallow\nInitiation',
    'Soft Palate\nElevation'
]
paper_replace_dict = {
    i:j for i,j in zip(action_names,paper_action_names)
}

In [23]:
def result_plot(video_data, result_dict, action_num=7, video_name=None, label_dict=None, start=0, end=-1):
    '''
    plot  C*T (action category * Time) matrix with results using seaborn heatmap
    video_data: np.array (T,H,W,3)
    result_dict: 
                {
                    $video_name:[
                        {
                            segment: [start_time, end_time],
                            label: action_name,
                            score: confidence score
                        }
                    ]
                }
    '''
    T = video_data.shape[0]  # Number of frames
    C = action_num  # Number of action categories
    fps = 29.97002997002997
    # Initialize the C*T matrix with zeros
    result_matrix = np.zeros((C, T))
    
    # If video_name is not provided, use the first key in result_dict
    if video_name is None:
        video_name = list(result_dict.keys())[0]
    
    # Fill the matrix with confidence scores where actions are predicted
    for prediction in result_dict[video_name]:
        start_time = prediction['segment'][0]
        end_time = prediction['segment'][1]
        label = prediction['label']
        # Map the label (str) to its corresponding ID (int)
        if label in label_dict:
            label_id = label_dict[label]
        else:
            # If the label is not in the ground truth, skip it or assign it to a default ID
            continue  # or assign to a default ID, e.g., label_id = action_num - 1
        score = prediction['score']
        
        # Ensure the times are within the video length
        start_time = max(0, min(start_time, T-1))
        end_time = max(0, min(end_time, T-1))
        start_frame = round(fps * start_time)
        end_frame = round(fps * end_time)
        # Fill the matrix with the confidence score for the predicted action
        result_matrix[label_id, start_frame:end_frame+1] += score
    
    # Plot the matrix using Seaborn
    plt.figure(figsize=(12, 6))
    sns.heatmap(result_matrix[:,start:end])
    plt.xlabel('Time (frames)')
    plt.ylabel('Action Category')
    plt.title(f'Predicted Action Matrix for {video_name}')
    plt.show()
    plt.close()

In [24]:
start=00
end=1200

In [25]:
gt_plot(video_data, ground_truth, video_name=video_name, start=start, end=end)

In [26]:
import numpy as np
from collections import defaultdict

def compute_iou(segment_a, segment_b):
    a_start, a_end = segment_a
    b_start, b_end = segment_b
    intersection_start = max(a_start, b_start)
    intersection_end = min(a_end, b_end)
    if intersection_start >= intersection_end:
        return 0.0
    intersection = intersection_end - intersection_start
    a_length = a_end - a_start
    b_length = b_end - b_start
    union = a_length + b_length - intersection
    return intersection / union if union != 0 else 0.0

def compute_ap(tp, fp, num_gt):
    if num_gt == 0:
        return 0.0
    tp_cum = np.cumsum(tp).astype(np.float32)
    fp_cum = np.cumsum(fp).astype(np.float32)
    recall = tp_cum / num_gt
    precision = tp_cum / (tp_cum + fp_cum + np.finfo(np.float32).eps)
    
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    
    i = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])
    return ap

def calculate_mAP_pervideo(ground_truth, results_dict):
    """Calculate mAP for temporal action localization task on video-level and return dict with {$video_name:$map}
    ground_truth: labels dict with standard format
        {
            $video_name:{
                annotataion:
                [
                    {
                        segment: [start_time, end_time],
                        label: action_name
                    },...
                ]
            }
        }
    results_dict: dict with method name as key, prediction dict as value
        {
            Ours: {
                $video_name:[
                    {
                        segment: [start_time, end_time],
                        label: action_name,
                        score: confidence score
                    }
                ]
            }
        }
    """
    final_result = {}
    for method_name in results_dict:
        method_map = {}
        predictions_dict = results_dict[method_name]
        for video_name in ground_truth:
            gt_entries = ground_truth[video_name]['annotations']
            gt_classes = defaultdict(list)
            for entry in gt_entries:
                gt_classes[entry['label']].append(entry['segment'])
            
            pred_entries = predictions_dict.get(video_name, [])
            pred_classes = defaultdict(list)
            for pred in pred_entries:
                label = pred['label']
                pred_classes[label].append((pred['segment'], pred['score']))
            
            for label in pred_classes:
                pred_classes[label].sort(key=lambda x: -x[1])
            
            aps = []
            for label in gt_classes:
                gt_segments = gt_classes[label]
                num_gt = len(gt_segments)
                pred_list = pred_classes.get(label, [])
                pred_segments = [seg for seg, _ in pred_list]
                
                tp = []
                fp = []
                used = [False] * num_gt
                
                for seg in pred_segments:
                    best_iou = -1.0
                    best_idx = -1
                    for i, gt_seg in enumerate(gt_segments):
                        if not used[i]:
                            iou = compute_iou(seg, gt_seg)
                            if iou > best_iou:
                                best_iou = iou
                                best_idx = i
                    if best_iou >= 0.5 and best_idx != -1:
                        tp.append(1)
                        fp.append(0)
                        used[best_idx] = True
                    else:
                        tp.append(0)
                        fp.append(1)
                
                ap = compute_ap(tp, fp, num_gt)
                aps.append(ap)
            
            mAP = np.mean(aps) if aps else 0.0
            method_map[video_name] = mAP
        final_result[method_name] = method_map
    return final_result

In [27]:
def cal_max_conf_action_overlap(results, gt):
    """calculate the most confidence actions' overlap with gt across all action categories
         on video-level and return dict with {$video_name:$map}
    ground_truth: labels dict with standard format
        {
            $video_name:{
                annotataion:
                [
                    {
                        segment: [start_time, end_time],
                        label: action_name
                    },...
                ]
            }
        }
    results_dict: dict with method name as key, prediction dict as value
        {
            Ours: {
                $video_name:[
                    {
                        segment: [start_time, end_time],
                        label: action_name,
                        score: confidence score
                    }
                ]
            }
        }
    """
    return 

In [28]:
map_dict=calculate_mAP_pervideo(ground_truth, results_dict)

In [29]:
video_names = list(result.keys())
filter_map_dict = {}
for video_name in video_names:
    if map_dict['Ruan et al. (A2Net)'][video_name] < map_dict['Ours (ActionMamba)'][video_name]:
        filter_map_dict[video_name] = {
            'Ruan et al (A2Net)': map_dict['Ruan et al. (A2Net)'][video_name],
            'Ours (ActionMamba)': map_dict['Ours (ActionMamba)'][video_name],
            'delta': map_dict['Ours (ActionMamba)'][video_name] - map_dict['Ruan et al. (A2Net)'][video_name]
        }
len(filter_map_dict)

115

In [30]:
[i for i in sorted(results_dict['Ruan et al. (A2Net)'][video_name],key=lambda x:x['score'],reverse=True) if i['label']=='HyoidExercise'][:4]

[{'segment': [15.755, 16.639], 'score': 0.9999, 'label': 'HyoidExercise'}]

In [31]:
[i for i in sorted(results_dict['Ours (ActionMamba)'][video_name],key=lambda x:x['score'],reverse=True) if i['label']=='HyoidExercise'][:4]

[{'score': 0.5382468700408936,
  'segment': [15.633552551269531, 16.6684513092041],
  'label': 'HyoidExercise'},
 {'score': 0.15117141604423523,
  'segment': [15.454890251159668, 16.766510009765625],
  'label': 'HyoidExercise'},
 {'score': 0.1161009818315506,
  'segment': [13.865527153015137, 14.828765869140625],
  'label': 'HyoidExercise'},
 {'score': 0.0693897008895874,
  'segment': [14.295742988586426, 16.0039119720459],
  'label': 'HyoidExercise'}]

In [32]:
import pandas as pd

# Assuming filter_map_dict is already created as per your code
# Transform the dictionary into a DataFrame
df = pd.DataFrame.from_dict(filter_map_dict, orient='index')

# Reset the index to make the video names a column
df.reset_index(inplace=True)
df.rename(columns={'index': 'Video Name'}, inplace=True)

In [33]:
df.sort_values('delta')[::-1]

,Video Name,Ruan et al (A2Net),Ours (ActionMamba),delta
81,7_272.0_2021062302_nie4fang1_jian4kang1cha2ti3...,0.000000,0.821428,8.214285e-01
42,8_304.0_2021062302_nie4fang1_jian4kang1cha2ti3...,0.000000,0.814286,8.142856e-01
48,9_304.0_2021062302_nie4fang1_jian4kang1cha2ti3...,0.000000,0.793233,7.932330e-01
97,4_48.0_2021062405_zhong1li4_jian4kang1cha2ti3_...,0.000000,0.761905,7.619047e-01
62,9_312.0_2021062302_nie4fang1_jian4kang1cha2ti3...,0.000000,0.757936,7.579364e-01
...,...,...,...,...
108,4_96.0_2021062302_nie4fang1_jian4kang1cha2ti3_...,0.857143,0.880952,2.380952e-02
44,10_272.0_2021062304_xie4cheng2si4_jian4kang1ch...,0.714286,0.734694,2.040818e-02
16,1_0.0_2021062302_nie4fang1_jian4kang1cha2ti3_2...,0.857143,0.875000,1.785714e-02
47,6_112.0_2021062302_nie4fang1_jian4kang1cha2ti3...,0.857143,0.875000,1.785714e-02


In [34]:
videos = df.sort_values('delta')[::-1]['Video Name'].to_list()

In [50]:
def plot_frames_and_comparsions(video_root,
                                select_index_or_video_name,
                                frame_row_size,
                                ground_truth,
                                results_dict,
                                video_frames_num=12,
                                filter_actions=None,
                                start_time=0,
                                end_time=64,
                                conf=0.5,
                                topk=1,
                                plot_rgb_frames=False,
                                fps = 29.97002997002997,
                                save_dir=None,
                                paper_replace_dict=None,
                                alpha=0.7,
                                colors=None):
    '''Plot the RGB frames, model results and ground-truth alongside time dimension
    Detail:
        - Plot the sampled RGB frames on top in first row when `plot_rgb_frames` is True, 
        subplot:
            - Plot the ground truth with filled rectangle (top)
            - Plot the different method results filled rectangle with different colors (position down by methods)
        - The same action category are in the same subplot
    Params:
        video_root: video directory root
        select_index_or_video_name: the selected video for ploting
        frame_row_size: the row height for rgb frames in the figure
        ground_truth: labels dict with standard format
            {
                $video_name:{
                    annotataion:
                    [
                        {
                            segment: [start_time, end_time],
                            label: action_name
                        },...
                    ]
                }
            }
        results_dict: dict with method name as key, prediction dict as value
            {
                Ours: {
                    $video_name:[
                        {
                            segment: [start_time, end_time],
                            label: action_name,
                            score: confidence score
                        }
                    ]
                }
            }
        dur: video total duration
        filter_actions: the actions want to plot, plot all when none
        start_time: the start time to plot
        end_time: the end time to plot
        conf: confidence threshold
        topk: only plot topk proposal(s) among same category action
        fps: video fps
        paper_replace_dict: formal action name dict (label_action: formal action)
    '''
    if isinstance(select_index_or_video_name, int):
        select_index = select_index_or_video_name
        video_name = list(results_dict[list(results_dict.keys())[0]].keys())[select_index]
    else:
        video_name = select_index_or_video_name
    if plot_rgb_frames:
        video_data = skvideo.io.vread(os.path.join(video_root, video_name+'.avi'))
    # video_data: numpy array store the rgb frames (T is frame-level)
    gt = ground_truth[video_name]['annotations']
    label_dict = get_label_dict_from_gt(ground_truth)
    action_names = list(label_dict.keys())
    if filter_actions is not None and len(filter_actions) > 0:
        gt = [item for item in gt if item['label'] in filter_actions]
    method_names = list(results_dict.keys())
    # filter results
    method_results = {}
    for method in method_names:
        if video_name not in results_dict[method]:
            raise KeyError(f"{video_name} not found in method: {method}")
        tmplist = results_dict[method][video_name]
        # instance number
        instance_num = len(gt)//len(label_dict)
        # filter actions
        if filter_actions is not None and len(filter_actions) > 0:
            tmplist = [
                i for i in tmplist
                if i['label'] in filter_actions and
                i['score'] > conf
            ]
        else:
            tmplist = [
                i for i in tmplist
                if i['score'] > conf
            ]
            
        # seperate with action category
        tmpdict = {}
        for action in action_names:
            if action not in tmpdict:
                tmpdict[action] = [i for i in tmplist if i['label'] == action]
        for action in action_names:
            if action in tmpdict and topk is not None:
                tmpdict[action] = tmpdict[action][:topk]
            elif action in tmpdict:
                tmpdict[action] = tmpdict[action][:instance_num]
        method_results[method] = tmpdict
    
    # Determine number of rows
    rows = len(action_names)
    if plot_rgb_frames:
        rows += 1
    total_bands = 1 + len(results_dict)  # 1 for GT, rest for methods
    band_height = 1.0 / total_bands

    fig = plt.figure(figsize=(12, rows + frame_row_size))
    gs = gridspec.GridSpec(rows, 1, height_ratios=([frame_row_size] if plot_rgb_frames else []) + [1] * len(action_names),
                           )
    colors = plt.cm.get_cmap('tab10', len(action_names)) if colors is None else colors
    if isinstance(colors, list):
        method_colors = {method: colors[i] for i, method in enumerate(method_names)}
    elif isinstance(colors, dict):
        method_colors = {method: colors[i] for i, method in enumerate(method_names)}
    else:
        method_colors = {method: colors(i) for i, method in enumerate(method_names)}
    current_row = 0

    if plot_rgb_frames and video_data is not None:
        total_frames = video_data.shape[0]
        start_frame = max(1, int(start_time * fps))
        end_frame = min(total_frames - 1, int(end_time * fps))
        num_sample_frames = video_frames_num
        frame_indices = np.linspace(start_frame, end_frame, num=num_sample_frames, dtype=int)
        gs0 = gridspec.GridSpecFromSubplotSpec(1, num_sample_frames, subplot_spec=gs[current_row],wspace=0)
        for i, fidx in enumerate(frame_indices):
            if fidx >= total_frames:
                continue
            # fig.subplots_adjust(bottom=0.01)
            ax = fig.add_subplot(gs0[i])
            ax.imshow(video_data[fidx])
            ax.axis('off')
        current_row += 1
        plt.subplots_adjust(bottom=0.01)  # 缩小底部边距

    for idx, action in enumerate(action_names):
        ax = fig.add_subplot(gs[current_row + idx])
        ax.set_xlim(start_time, end_time)
        ax.set_xticklabels(ax.get_xticklabels(), fontsize=14)
        ax.set_ylim(0, 1)
        ax.set_yticks([])
        if paper_replace_dict is not None:
            ax.set_ylabel(paper_replace_dict[action], rotation=0, labelpad=60, va='center', fontsize=18)
        else:
            ax.set_ylabel(action, rotation=0, labelpad=40, va='center',fontsize=18)

        # Plot Ground Truth
        for anno in gt:
            if anno['label'] == action:
                start = max(anno['segment'][0], start_time)
                end = min(anno['segment'][1], end_time)
                if start >= end:
                    continue
                y_gt = 1 - band_height
                rect = Rectangle((start, y_gt), end - start, band_height, color='green', alpha=alpha)
                ax.add_patch(rect)

        # Plot Methods
        for method_idx, method in enumerate(method_names):
            color = method_colors[method]
            for pred in method_results[method][action]:
                start = max(pred['segment'][0], start_time)
                end = min(pred['segment'][1], end_time)
                if start >= end:
                    continue
                y_method = 1 - (method_idx + 2) * band_height
                rect = Rectangle((start, y_method), end - start, band_height, color=color, alpha=alpha)
                ax.add_patch(rect)

        if idx == len(action_names) - 1:
            ax.set_xlabel('Time (seconds)', fontsize=20)
        else:
            ax.set_xticklabels([])

    legend_handles = [Rectangle((0, 0), 1, 1, color='green', alpha=alpha, label='Ground Truth')]
    legend_handles.extend([Rectangle((0, 0), 1, 1, color=method_colors[method], alpha=alpha, label=method)
                          for method in method_names])
    if plot_rgb_frames:
        fig.legend(handles=legend_handles, loc='upper right', 
                fontsize=16,
                bbox_to_anchor=(1, 0.9), 
                bbox_transform=fig.transFigure,  # 坐标系基于整个画布
                )
    else:
        fig.legend(handles=legend_handles, loc='upper right', fontsize=16)
    plt.tight_layout()
    if save_dir:
        os.makedirs(f'{save_dir}', exist_ok=True)
        # fig.savefig(f'{save_dir}/result-visual-{select_index_or_video_name}.svg', dpi=600)
        plt.savefig(f'{save_dir}/result-visual-{select_index_or_video_name}.png', dpi=600)
        print(f'save to {save_dir}/result-visual-{select_index_or_video_name}.png')
    plt.close()

In [52]:

plot_frames_and_comparsions(video_root, 
                            # select_index_or_video_name='3_112.0_2021063001_li3wei3qiang2_jian4kang1cha2ti3_2021_06_30_114059_64', 
                            # select_index_or_video_name='2_96.0_2021063001_li3wei3qiang2_jian4kang1cha2ti3_2021_06_30_114059_64', 
                            select_index_or_video_name='8_96.0_2021082501_liu2meng2_jian4kang1cha2ti3_2021_08_25_152802_32', 
                            frame_row_size=1,
                            video_frames_num=12,
                            ground_truth=ground_truth,
                            results_dict=results_dict,
                            plot_rgb_frames=True,
                            topk=None,
                            conf=0.2,
                            paper_replace_dict=paper_replace_dict,
                            save_dir='debug',
                            start_time=18,
                            end_time=38,
                            alpha=0.7,
                            # colors=["#33A02C","#E31A1C","#1F78B4","#FB9A99"],
                            )

/tmp/ipykernel_1561029/278094799.py:119: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colors = plt.cm.get_cmap('tab10', len(action_names)) if colors is None else colors
/tmp/ipykernel_1561029/278094799.py:148: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), fontsize=14)
/tmp/ipykernel_1561029/278094799.py:148: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), fontsize=14)
/tmp/ipykernel_1561029/278094799.py:148: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabel

save to debug/result-visual-8_96.0_2021082501_liu2meng2_jian4kang1cha2ti3_2021_08_25_152802_32.png


In [39]:
from tqdm import tqdm
video_name = videos[14]
for i in tqdm(videos):
    try:
        plot_frames_and_comparsions(video_root, 
                                    select_index_or_video_name=i, 
                                    frame_row_size=2,
                                    ground_truth=ground_truth,
                                    results_dict=results_dict,
                                    # plot_rgb_frames=True,
                                    topk=None,
                                    conf=0.2,
                                    paper_replace_dict=paper_replace_dict,
                                    save_dir='3methods-visual',
                                    )
    except KeyError as e:
        ...

  0%|          | 0/115 [00:00<?, ?it/s]/tmp/ipykernel_34361/949511034.py:116: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colors = plt.cm.get_cmap('tab10', len(action_names))
/tmp/ipykernel_34361/949511034.py:138: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), fontsize=14)
/tmp/ipykernel_34361/949511034.py:138: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), fontsize=14)
/tmp/ipykernel_34361/949511034.py:138: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabe